# Implement RMS Normalization from Scratch
### Problem Statement
RMS Normalization (RMSNorm), introduced in [Zhang & Sennrich 2019](https://arxiv.org/abs/1910.07467), is a simplified alternative to Layer Normalization. Instead of computing both mean and variance, RMSNorm only computes the **root mean square** of the activations.

This makes it ~10-15% faster than LayerNorm while achieving comparable performance. RMSNorm is the default normalization in modern LLMs including Llama, Llama 2/3, Mistral, and Gemma.

---

### Requirements

1. **RMS Computation**
   - Compute: `rms = sqrt(mean(x²) + eps)`
   - Normalize: `x_norm = x / rms`

2. **Learnable Scale Parameter**
   - Apply a learnable `gamma` (initialized to ones): `output = x_norm * gamma`
   - Note: RMSNorm has **no bias** and **no mean centering** — this is the key difference from LayerNorm.

3. **Validate**
   - Output shape must match input shape.
   - Compare behavior with a manual computation.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Support arbitrary input shapes `(..., dim)` — normalize over the last dimension.
- ✅ Include `eps` for numerical stability.

---

<details>
  <summary>💡 Hint</summary>

  - `x.pow(2).mean(dim=-1, keepdim=True)` gives you the mean of squared values.
  - `torch.sqrt(...)` to get the RMS.
  - The scale parameter `gamma` is `nn.Parameter(torch.ones(dim))`.
  - Unlike LayerNorm, there is no mean subtraction step.

</details>

---

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class RMSNorm(nn.Module):
    """
    Implements Root Mean Square Layer Normalization.
    """
    def __init__(self, dim: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))  # gamma

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (Tensor): Input tensor of shape (..., dim)

        Returns:
            Tensor: Normalized tensor of same shape
        """
        # Compute RMS: sqrt(mean(x^2) + eps)
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

        # Normalize and scale
        return (x / rms) * self.scale

## 📚 RMSNorm vs LayerNorm

### LayerNorm formula
```
LayerNorm(x) = gamma * (x - mean(x)) / sqrt(var(x) + eps) + beta
```
- Computes **mean** and **variance**
- Has both **gamma** (scale) and **beta** (bias)
- Re-centers the distribution (subtracts mean)

### RMSNorm formula
```
RMSNorm(x) = gamma * x / sqrt(mean(x²) + eps)
```
- Only computes **root mean square** (no mean subtraction)
- Has only **gamma** (no bias)
- Simpler and faster — skips the mean computation and bias addition

### Why does removing the mean centering work?

The authors showed that the **re-scaling invariance** (not the re-centering) is the key property that makes normalization effective for training stability. Removing mean centering:
- Reduces computation by ~10-15%
- Has negligible impact on model quality
- Is particularly beneficial for large models where normalization is called many times

### Where it's used

| Model | Normalization | Position |
|-------|--------------|----------|
| GPT-2/3 | LayerNorm | Post-attention, post-FFN |
| Llama 1/2/3 | RMSNorm | **Pre**-attention, **Pre**-FFN |
| Mistral | RMSNorm | Pre-attention, Pre-FFN |
| Gemma | RMSNorm | Pre-attention, Pre-FFN |

In [ ]:
# Test
torch.manual_seed(42)
x = torch.randn(3, 4, 8)  # (batch, seq_len, dim)
rmsnorm = RMSNorm(dim=8)
out = rmsnorm(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == x.shape, "Output shape mismatch"

# Manual verification
rms_manual = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + 1e-8)
expected = x / rms_manual  # gamma is all ones initially
assert torch.allclose(out, expected, atol=1e-6)
print("\n✅ RMSNorm implementation is correct.")